# Customer Segmentation – RFM + K-Means Clustering

**Team:** Hamza Amhidi & Mouaad Kaddouri  
**Dataset:** [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)

> [!WARNING]
> **⚠️ IMPORTANT FOR GOOGLE COLAB USERS:**  
> You **must** run **Phase 0: Setup and Environment Preparation** (the first code cell below) before executing any other cells in this notebook. Otherwise, imports will fail with `ModuleNotFoundError` since the repository code must be cloned into the Colab workspace first.

---

## Table of Contents

1. [Phase 0: Setup and Environment Preparation](#Phase-0:-Setup-and-Environment-Preparation)
2. [Phase 1: Data Ingestion & Quality Auditing](#Phase-1:-Data-Ingestion-&-Quality-Auditing)
3. [Phase 2: Data Wrangling & Cleaning](#Phase-2:-Data-Wrangling-&-Cleaning)
4. [Phase 3: Feature Engineering](#Phase-3:-Feature-Engineering)
5. [Phase 3b: Relational SQL Database Storage & Queries](#Phase-3b:-Relational-SQL-Database-Storage-&-Queries)
6. [Phase 3c: Exploratory Data Analysis (EDA)](#Phase-3c:-Exploratory-Data-Analysis-(EDA))
7. [Phase 4: Algorithm Selection & Justification](#Phase-4:-Algorithm-Selection-&-Justification)
8. [Phase 5: Machine Learning Clustering & Segment Profiling](#Phase-5:-Machine-Learning-Clustering-&-Segment-Profiling)
9. [Phase 6: Model Card & Ethical Reflection](#Phase-6:-Model-Card-&-Ethical-Reflection)
10. [Phase 7: Final Report & Oral Presentation Outline](#Phase-7:-Final-Report-&-Oral-Presentation-Outline)

## Phase 0: Setup and Environment Preparation

In [ ]:
import os
import sys

# Setup Google Colab workspace environments
try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    if not os.path.exists('Customer-Segmentation'):
        # Clone the repository directly into the Colab environment
        !git clone https://github.com/echoenvoy/Customer-Segmentation.git
    os.chdir('Customer-Segmentation')

# Local/Binder environment setup: ensure working directory is repository root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Add the src directory to the system path
sys.path.insert(0, os.path.abspath('src'))

# Set plotting themes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## Phase 1: Data Ingestion & Quality Auditing

First, we acquire the Online Retail dataset (downloading and converting if not present) and perform a comprehensive quality audit of the raw data.

In [ ]:
import load_describe_data

# Download dataset (skips if already downloaded)
load_describe_data.download_dataset()

# Load raw transactions (fast CSV loading)
RAW_CSV = "data/raw/online_retail_raw.csv"
if os.path.exists(RAW_CSV):
    df_raw = pd.read_csv(RAW_CSV, dtype={"CustomerID": str, "InvoiceNo": str})
    df_raw["InvoiceDate"] = pd.to_datetime(df_raw["InvoiceDate"])
else:
    df_raw = load_describe_data.load_data(load_describe_data.RAW_FILE)

# Run data audit
load_describe_data.audit(df_raw)

## Phase 2: Data Wrangling & Cleaning

We clean the raw data by:
1. Removing exact duplicate rows.
2. Dropping records with missing `CustomerID`.
3. Excluding invoice cancellations (which are prefixed with 'C').
4. Dropping transactions with non-positive quantity or price.
5. Calculating a calculated `Revenue` column.

In [ ]:
import clean_data

# Clean raw data and save output
df_clean = clean_data.clean(df_raw)
os.makedirs("data/processed", exist_ok=True)
df_clean.to_csv("data/processed/transactions_clean.csv", index=False)

## Phase 3: Feature Engineering

We aggregate the transaction-level records to build 7 behavioral customer-level features:
- **Recency:** Days since last purchase.
- **Frequency:** Unique invoices (orders).
- **Monetary:** Sum of revenue.
- **AvgOrderValue:** Monetary / Frequency.
- **UniqueProducts:** Count of distinct stock codes purchased.
- **ActiveDays:** Days between first and last purchase.
- **CancellationRate:** Fraction of cancelled invoices (computed at unique invoice level).

In [ ]:
import features as feat

# Build customer features
df_feat = feat.build_features(df_clean, df_raw)
df_feat.to_csv("data/processed/customer_features.csv", index=False)
print(f"Customer features built: {df_feat.shape[0]} customers, {df_feat.shape[1]} features")
df_feat.describe().round(2)

## Phase 3b: Relational SQL Database Storage & Queries

We load the clean transactions and engineered features into a local SQLite database (`retail.db`) and run analysis.

In [ ]:
import store_sql

# SQLite storage loader
store_sql.load_to_sqlite(df_clean, df_feat)

# Run analytical queries
store_sql.run_all_queries()

## Phase 3c: Exploratory Data Analysis (EDA)

Let's display some of the key insights from our transaction and customer metrics.

In [ ]:
# Display Top Countries by Revenue (excluding UK)
rev = df_clean[df_clean["Country"] != "United Kingdom"].groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 4))
sns.barplot(x=rev.values, y=rev.index, color="steelblue")
plt.title("Top 10 Countries by Revenue (excl. UK)")
plt.xlabel("Total Revenue (£)")
plt.ylabel("")
plt.show()

# Display RFM Pairplot (with log-transformed frequency & monetary)
vis = df_feat[["Recency", "Frequency", "Monetary"]].copy()
vis["Frequency"] = np.log1p(vis["Frequency"])
vis["Monetary"] = np.log1p(vis["Monetary"])
vis = vis.rename(columns={"Frequency": "log(Frequency)", "Monetary": "log(Monetary)"})
g = sns.pairplot(vis, plot_kws={"alpha": 0.3, "s": 8}, diag_kind="hist")
g.figure.suptitle("RFM Pairplot (Log-Transformed Frequency & Monetary)", y=1.02)
plt.show()

## Phase 4: Algorithm Selection & Justification

### 1. Candidate Algorithms Comparison

To choose the best model for customer segmentation, we compared four major clustering algorithms on our preprocessed RFM + Cancellation features:

| Algorithm | Working Principle | Key Assumptions | Strengths | Weaknesses | Suitability for RFM |
|---|---|---|---|---|---|
| **K-Means** | Partitions data into $K$ spherical clusters by minimizing WCSS. | Clusters are spherical and of similar density/size. | Highly scalable, simple to compute, and easy to interpret. | Vulnerable to outliers and struggles with complex shapes. | **High** (after log1p scaling and RobustScaler scaling). |
| **DBSCAN** | Density-based algorithm grouping close neighbors. | Clusters are separated by lower density regions. | Automatically detects noise/outliers and finds arbitrary shapes. | Sensitive to parameter tuning; struggles with varying densities. | **Medium** (handles outliers well but merges overlapping customer density). |
| **GMM (Gaussian Mixture Model)** | Soft-clustering model assuming multivariate Gaussian mixtures. | Data belongs to a mixture of normal distributions. | Flexible cluster shapes; outputs soft membership probabilities. | Sensitive to initialization and computationally expensive. | **Medium** (adds complexity without clear business benefits). |
| **Agglomerative Hierarchical** | Bottom-up clustering merging nearest pairs sequentially. | Distance hierarchy represents true clustering relationships. | Produces a dendrogram; does not assume spherical clusters. | High computational complexity ($O(N^2 \log N)$). | **Low** (too slow for larger transactional databases). |

### 2. Evaluation Metrics Justification

To determine the optimal number of clusters ($K$) and evaluate the separation quality, we track three standard metrics:

1.  **Silhouette Coefficient (Primary):**
    *   *Formula:* $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$
    *   *Range:* $[-1, 1]$ (where $+1$ indicates highly distinct clusters, $0$ indicates overlapping clusters).
    *   *Justification:* Measures cluster cohesion and separation, giving a direct index of segment distinctness.
2.  **Davies-Bouldin Index (Secondary):**
    *   *Formula:* Average similarity of each cluster with its most similar counterpart based on centroids.
    *   *Range:* $[0, \infty)$ (lower values close to $0$ indicate tight, well-separated partitions).
    *   *Justification:* Rewards tight clusters, helping validate the silhouette score.
3.  **Calinski-Harabasz Score (Variance Ratio):**
    *   *Formula:* Ratio of between-cluster variance to within-cluster variance.
    *   *Justification:* High scores indicate well-defined, highly separated cluster boundaries.

### 3. Final Algorithm Selection

We chose **K-Means Clustering** because:
1.  **Interpretability:** Centroids represent the "average customer" of that segment, which translates directly into clear marketing personas.
2.  **Preprocessing Mitigation:** By preprocessing features using a `log1p` transformation and a `RobustScaler`, we stabilized the variance and ignored extreme outliers, allowing K-Means to partition the data cleanly.
3.  **Efficiency:** Runs in linear time $O(I \cdot K \cdot N \cdot D)$, ensuring immediate execution and scalability.

## Phase 5: Machine Learning Clustering & Segment Profiling

We train our clustering model:
1. Log-transform right-skewed features (`Recency`, `Frequency`, `Monetary`, `AvgOrderValue`, `UniqueProducts`).
2. Scale features using `RobustScaler` to ignore remaining outlier skew.
3. Run K-Means with $K=2 \dots 8$ to evaluate optimal cluster count.
4. Train final model with the optimal $K=3$.
5. Profile clusters and uniquely map them to: **VIP**, **Loyal**, and **At-Risk** segments.

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import train_cluster_models as tcm

# 1. Select & transform features
CLUSTER_FEATURES = ["Recency", "Frequency", "Monetary", "AvgOrderValue", "UniqueProducts", "ActiveDays", "CancellationRate"]
X = df_feat[CLUSTER_FEATURES].copy()
skewed = ["Recency", "Frequency", "Monetary", "AvgOrderValue", "UniqueProducts"]
for col in skewed:
    X[col] = np.log1p(X[col])

# 2. Scale features
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# 3. Run K Search
print("Evaluating K=2 to 8:")
best_k = tcm.find_optimal_k(X_scaled)

# 4. Fit final model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_feat["Cluster"] = kmeans.fit_predict(X_scaled)

# 5. Map unique segments
df_clustered, profile = tcm.profile_clusters(df_feat)

# Show evaluation metrics
sil = silhouette_score(X_scaled, df_clustered["Cluster"])
dbi = davies_bouldin_score(X_scaled, df_clustered["Cluster"])
print(f"\nSilhouette Score: {sil:.4f}")
print(f"Davies-Bouldin Index: {dbi:.4f}")

# Display segment counts
print("\nSegment Counts:")
print(df_clustered.groupby(["Cluster", "Segment"]).size().reset_index(name="Customers"))

### Cluster Profiles (Normalized Feature Means)

Let's visualize the profile of each customer segment.

In [ ]:
# Display cluster profile heatmap
norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)
plt.figure(figsize=(10, 4))
sns.heatmap(norm.T, annot=profile.T.round(1), fmt="g", cmap="YlOrRd", linewidths=0.5)
plt.title("Cluster Profiles - Normalized Feature Means")
plt.xlabel("Cluster")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Phase 6: Model Card & Ethical Reflection

### 1. Model Card

#### Model Overview
*   **Model Name:** RFM Customer Segmentation Pipeline
*   **Model Type:** K-Means Clustering (unsupervised)
*   **Task:** Clustering customers into 3 behavioral segments (VIP, Loyal, At-Risk)
*   **Version:** 1.0.0
*   **Date:** June 17, 2026
*   **Trained By:** Hamza Amhidi & Mouaad Kaddouri

#### Intended Use
*   **Target Audience:** Marketing teams, sales managers, and retail analysts.
*   **Supported Decisions:** Targeted email marketing, loyalty program invitations, churn prevention campaigns, and promotional budgeting.
*   **Out of Scope:** Individual sales predictions, credit scoring, or automated pricing decisions.

#### Training Data
*   **Source:** [UCI Online Retail Dataset](https://archive.ics.uci.edu/dataset/352/online+retail) (transactions from 2010-12-01 to 2011-12-09).
*   **Size:** 4,338 unique customers (extracted from 392,692 cleaned purchase lines).
*   **Features Used:** `Recency`, `Frequency`, `Monetary`, `AvgOrderValue`, `UniqueProducts`, `ActiveDays`, `CancellationRate`.
*   **Preprocessing Summary:** `log1p` transformation of skewed columns, followed by `RobustScaler` scaling.

#### Performance Summary
*   **Optimal Clusters ($K$):** 3
*   **Silhouette Score:** 0.3093 (good cohesion and clear boundaries)
*   **Davies-Bouldin Index:** 1.2021 (strong cluster separation)

#### Limitations
*   **Time Sensitivity:** Customer behavior changes over time, meaning the pipeline must be run regularly (e.g. monthly) to capture shifts.
*   **Data Integrity:** The pipeline relies on accurate `CustomerID` values. Guest checkout transactions (missing IDs) are skipped.

---

### 2. Ethical Reflection

#### Data Privacy
*   **Anonymization:** The dataset only tracks transactions via numeric `CustomerID` and `InvoiceNo` fields. No personally identifiable information (PII) such as customer names, emails, credit card numbers, or street addresses is stored or processed.
*   **Security:** The SQLite database is stored locally (`retail.db`) and excluded from public version control to avoid database leak risks.

#### Algorithmic Bias & Fairness
*   **VIP Exclusion:** Using monetary metrics to categorize customers can lead to biases in customer support or promotions. Marketing teams must ensure that "Low-Value" customers are not completely ignored or mistreated, as they represent the highest volume of potential growth.
*   **Return-Rate Labeling:** Customers with a high `CancellationRate` are flagged. While useful for business logistics, labeling them "Risky VIPs" should not lead to punitive measures (such as blocking accounts) without human audit, as cancellations can occur due to defective products or shipping errors.

## Phase 7: Final Report & Oral Presentation Outline

Our upcoming final report and presentation slides will follow this structure:
1. **Introduction & Business Case:** Retail customer segmentation to optimize budget distribution and conversion rate.
2. **Data Wrangling:** Process to clean duplicates, handle cancellations, and filter guest checkouts.
3. **Behavioral Feature Engineering:** Deriving RFM features + active days and return risk metrics.
4. **Machine Learning Approach:** Preprocessing outliers (log-transform + median scaling) and training K-Means.
5. **Segment Profiles & Action Plan:** Custom strategies for VIPs (high-touch), Loyal (upselling), and At-Risk (win-back).
6. **Ethical Reflection:** Privacy safeguards and marketing equity reviews.